# Сегментація клієнтів за портфелем ТЗ — повний цикл навчання

**Порядок роботи**

1. **Запуск 1.** Виконай клітинки по черзі до кроку **8 (профілі кластерів)** включно.
   Надішли таблиці з кроків **7** (метрики по k) і **8** (профілі). Кластери поки називаються `CLUSTER_1 … CLUSTER_n`
   (нумерація за зростанням медіанної вартості портфеля).
2. Отримай назви й встав їх у словник `CLUSTER_NAMES` (крок 9).
3. **Запуск 2.** У налаштуваннях постав `USE_SAVED_MODEL = True` (модель з output/ не перенавчається,
   нумерація кластерів гарантовано та сама) і виконай усі клітинки (*Run All*).

Усі результати зберігаються в папку `output/`.

In [ ]:
import sys
import json
import importlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from IPython.display import display

import utils
importlib.reload(utils)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

print("python      ", sys.version.split()[0])
print("pandas      ", pd.__version__)
print("numpy       ", np.__version__)
print("scikit-learn", sklearn.__version__)

## 0. Налаштування

In [ ]:
DATA_PATH = Path(r'C:\repo\data-science\Nazar Hnatyshyn\RETAIL\(DS-1052) Vehicle market analysis\df_for_clustering.xlsx')
SHEET_NAME = 0                 # перший аркуш

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(exist_ok=True)

RANDOM_STATE = utils.RANDOM_STATE
BUSINESS_THRESHOLD = 1         # від скількох бізнес-ТЗ клієнт з легковиком іде в BUSINESS_VEHICLE_OWNERS

# KMeans
K_RANGE = range(5, 11)         # які k перебираємо
K_FINAL = None                 # None — автоматичний вибір; або число, напр. 7
RUN_K_SELECTION = True         # False — пропустити перебір k (тоді K_FINAL має бути задано)
USE_SAVED_MODEL = False        # True — взяти scaler + KMeans з output/ (запуск 2)
SAMPLE_SIZE = 50_000           # вибірка для перебору k
SILHOUETTE_SAMPLE = 10_000     # підвибірка для silhouette
N_SEEDS = 5                    # кількість seed для перевірки стабільності (ARI)

# Дерево правил
TREE_DEPTH = None              # None — вибір між 4 і 5 автоматично

print("Дані:  ", DATA_PATH, "| існує:", DATA_PATH.exists())
print("Output:", OUTPUT_DIR.resolve())

## 1. Завантаження даних

In [ ]:
raw = utils.load_raw(DATA_PATH, SHEET_NAME)
print(f"Рядків (ТЗ): {len(raw):,} | клієнтів: {raw['user_id'].nunique():,} | колонок: {raw.shape[1]}")

overview = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "non_null": raw.notna().sum(),
    "null_pct": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
})
display(overview)
display(raw.head())

## 2. Підготовка на рівні авто: класи ТЗ

In [ ]:
vehicles = utils.prepare_vehicle_data(raw)
print("Shape:", vehicles.shape)

class_counts = vehicles["VEHICLE_CLASS"].value_counts().to_frame("VEHICLES")
class_counts["PCT"] = (class_counts["VEHICLES"] / len(vehicles) * 100).round(2)
display(class_counts)

print("kind -> клас ТЗ (перевір, чи все розкладено правильно, особливо OTHER):")
kind_table = pd.crosstab(vehicles["KIND_STD"], vehicles["VEHICLE_CLASS"])
display(kind_table.loc[kind_table.sum(axis=1).sort_values(ascending=False).index])

## 3. Пальне: fuel_type → FUEL_CLASS

In [ ]:
fuel_table = (
    vehicles.assign(fuel_type=vehicles["fuel_type"].fillna("<порожнє>"))
    .groupby(["fuel_type", "FUEL_CLASS"]).size()
    .rename("VEHICLES").reset_index()
    .sort_values("VEHICLES", ascending=False)
)
display(fuel_table)

passenger = vehicles["IS_PASSENGER"].eq(1)
print("Розподіл класів пального серед легковиків:")
display(vehicles.loc[passenger, "FUEL_CLASS"].value_counts().to_frame("CARS"))

## 4. Причепи

In [ ]:
trailers = vehicles.loc[vehicles["IS_TRAILER"].eq(1)]
print(f"Причепів: {len(trailers):,}")
display(trailers["TRAILER_TYPE"].value_counts().to_frame("TRAILERS"))

trailer_table = (
    trailers.groupby(["BODY_STD", "TRAILER_TYPE"]).size()
    .rename("TRAILERS").reset_index()
    .sort_values("TRAILERS", ascending=False)
)
display(trailer_table)

print("Причепів без ваги:", int(vehicles["TRAILER_WITHOUT_WEIGHT"].sum()))
print("Житлових причепів (camper):", int(vehicles["IS_CAMPER_TRAILER"].sum()))

exceptions = vehicles.loc[
    vehicles["BRAND_STD"].isin(["OPEL", "FIAT", "FORD", "ОДИСЕЙ", "KOGEL"])
    & (vehicles["BODY_STD"].str.startswith(("ПР-", "Н/ПР-")))
    , ["BRAND_STD", "MODEL_STD", "KIND_STD", "BODY_STD", "total_weight", "VEHICLE_CLASS", "TRAILER_TYPE"]
]
print("Погоджені винятки:")
display(exceptions)

## 5. SUV, premium, вік

In [ ]:
cars = vehicles.loc[passenger]
print(f"Легковиків: {len(cars):,}")
print(f"SUV:        {int(cars['IS_SUV'].sum()):,} ({cars['IS_SUV'].mean() * 100:.1f}%)")
print(f"Premium:    {int(cars['IS_PREMIUM_BRAND'].sum()):,} ({cars['IS_PREMIUM_BRAND'].mean() * 100:.1f}%)")
print(f"EV:         {int(cars['IS_EV'].sum()):,} | HYBRID: {int(cars['IS_HYBRID'].sum()):,} | GAS: {int(cars['IS_GAS'].sum()):,}")
print(f"Авто >= $60k (luxury vehicle): {int(cars['IS_TIER5'].sum()):,}")
print(f"Modern premium авто: {int(cars['IS_MODERN_PREMIUM'].sum()):,}")
print(f"Рік 1900 (невідомий): {int(cars['PRODUCTION_YEAR_CLEAN'].isna().sum()):,}")

print("\nТоп-30 SUV-моделей:")
display(cars.loc[cars["IS_SUV"].eq(1)].groupby(["BRAND_STD", "MODEL_STD"]).size()
        .sort_values(ascending=False).head(30).to_frame("CARS"))

print("Топ-40 моделей, які НЕ визначені як SUV (перевір, чи немає серед них кросоверів):")
display(cars.loc[cars["IS_SUV"].eq(0)].groupby(["BRAND_STD", "MODEL_STD"]).size()
        .sort_values(ascending=False).head(40).to_frame("CARS"))

print("Вік легковиків:")
display(cars["VEHICLE_AGE_CLEAN"].describe().to_frame("AGE").T)

## 6. Агрегація до рівня клієнта

In [ ]:
AGE_MEDIAN = utils.compute_age_median(vehicles)
customers = utils.aggregate_customers(vehicles, AGE_MEDIAN)
print(f"Медіана віку для пропусків: {AGE_MEDIAN}")
print(f"Клієнтів: {len(customers):,} | ознак: {customers.shape[1]}")

bins = [-1, 0, 1, 2, 3, 4, 5, np.inf]
labels = ["0", "1", "2", "3", "4", "5", "6+"]
cars_dist = pd.cut(customers["PASSENGER_VEHICLE_COUNT"], bins=bins, labels=labels).value_counts().sort_index()
cars_dist = cars_dist.to_frame("CLIENTS")
cars_dist["PCT"] = (cars_dist["CLIENTS"] / len(customers) * 100).round(2)
print("Кількість легковиків на клієнта:")
display(cars_dist)

key_cols = ["PASSENGER_VEHICLE_COUNT", "TOTAL_VEHICLE_VALUE_USD", "MAX_VEHICLE_VALUE_USD", "AVG_VEHICLE_AGE",
            "PREMIUM_SHARE", "SUV_SHARE", "EV_SHARE", "HYBRID_SHARE", "GAS_SHARE", "BUSINESS_VEHICLE_COUNT", "TRAILER_SHARE"]
display(customers[key_cols].describe(percentiles=[0.05, 0.5, 0.95]).T)

flag_cols = [c for c in customers.columns if c.startswith("HAS_")]
display((customers[flag_cols].mean() * 100).round(2).to_frame("PCT_CLIENTS"))

## 7. Популяція KMeans і вибір кількості кластерів

KMeans навчається лише на клієнтах, яких не забрали правила ієрархії:
1–2 легковики, без бізнес-ТЗ, без авто ≥ $60k, без modern premium.

In [ ]:
masks_preview, _ = utils.rule_masks(customers, ev_rule=False, business_threshold=BUSINESS_THRESHOLD)
preview = pd.DataFrame([(name, int(mask.sum())) for name, mask in masks_preview], columns=["SEGMENT", "CLIENTS"])
tree_mask = utils.tree_population_mask(customers, ev_rule=False, business_threshold=BUSINESS_THRESHOLD)
preview.loc[len(preview)] = ["→ популяція KMeans / дерева", int(tree_mask.sum())]
preview["PCT"] = (preview["CLIENTS"] / len(customers) * 100).round(2)
print("Сегменти за правилами (попередньо, без EV-правила):")
display(preview)

In [ ]:
def build_population(ev_rule):
    mask = utils.tree_population_mask(customers, ev_rule=ev_rule, business_threshold=BUSINESS_THRESHOLD)
    return customers.loc[mask].copy()


km_population = build_population(ev_rule=False)
X_all = StandardScaler().fit_transform(km_population[utils.CLUSTER_FEATURES])
print(f"Популяція KMeans: {len(km_population):,} клієнтів, ознаки: {utils.CLUSTER_FEATURES}")

if RUN_K_SELECTION and not USE_SAVED_MODEL:
    rng = np.random.default_rng(RANDOM_STATE)
    sample_idx = rng.choice(len(X_all), size=min(SAMPLE_SIZE, len(X_all)), replace=False)
    X_sample = X_all[sample_idx]

    rows = []
    for k in K_RANGE:
        labels_by_seed = [
            KMeans(n_clusters=k, n_init=5, random_state=RANDOM_STATE + seed).fit_predict(X_sample)
            for seed in range(N_SEEDS)
        ]
        labels = labels_by_seed[0]
        aris = [
            adjusted_rand_score(labels_by_seed[i], labels_by_seed[j])
            for i in range(N_SEEDS) for j in range(i + 1, N_SEEDS)
        ]
        sils = [
            silhouette_score(X_sample, labels, sample_size=min(SILHOUETTE_SAMPLE, len(X_sample)),
                             random_state=RANDOM_STATE + s)
            for s in range(5)
        ]
        sizes = np.bincount(labels, minlength=k) / len(labels) * 100
        rows.append({
            "k": k,
            "SILHOUETTE_MEAN": np.mean(sils),
            "SILHOUETTE_STD": np.std(sils),
            "DAVIES_BOULDIN": davies_bouldin_score(X_sample, labels),
            "ARI_MEAN": np.mean(aris),
            "ARI_MIN": np.min(aris),
            "MIN_CLUSTER_PCT": sizes.min(),
            "MAX_CLUSTER_PCT": sizes.max(),
            "SIZES_PCT": ", ".join(f"{s:.1f}" for s in sorted(sizes, reverse=True)),
        })
        print(f"k={k} готово")

    k_table = pd.DataFrame(rows).set_index("k")
    k_table.to_csv(OUTPUT_DIR / "k_selection.csv")
    display(k_table)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    k_table["SILHOUETTE_MEAN"].plot(ax=axes[0], marker="o", title="Silhouette (вище — краще)")
    k_table["DAVIES_BOULDIN"].plot(ax=axes[1], marker="o", title="Davies-Bouldin (нижче — краще)")
    k_table["ARI_MEAN"].plot(ax=axes[2], marker="o", title="Стабільність ARI (вище — краще)")
    axes[2].axhline(0.95, color="grey", linestyle="--")
    plt.tight_layout()
    plt.show()
else:
    k_table = None
    print("Перебір k пропущено.")

In [ ]:
# Вибір K: найкращий silhouette серед стабільних (ARI >= 0.95) k без дрібних кластерів (< 1%)
if USE_SAVED_MODEL:
    saved_model = joblib.load(OUTPUT_DIR / "kmeans_model.joblib")
    K_CHOSEN = saved_model["k"]
    print(f"K з збереженої моделі: {K_CHOSEN}")
elif K_FINAL is not None:
    K_CHOSEN = int(K_FINAL)
    print(f"K задано вручну: {K_CHOSEN}")
else:
    candidates = k_table[(k_table["ARI_MEAN"] >= 0.95) & (k_table["MIN_CLUSTER_PCT"] >= 1.0)]
    if len(candidates):
        K_CHOSEN = int(candidates["SILHOUETTE_MEAN"].idxmax())
        print(f"Автоматично обрано K = {K_CHOSEN} (найкращий silhouette серед стабільних)")
    else:
        K_CHOSEN = int(k_table["ARI_MEAN"].idxmax())
        print(f"Немає k з ARI >= 0.95 — обрано найстабільніший K = {K_CHOSEN}")

## 8. Фінальний KMeans і профілі кластерів

Якщо KMeans не виділяє окремий EV-кластер, EV_ADOPTERS задається правилом (EV_SHARE = 1),
а електромобілістів прибираємо з популяції й перенавчаємо модель.
Кластери нумеруються за зростанням медіанної вартості портфеля: CLUSTER_1 — найдешевший.

In [ ]:
def fit_kmeans(population, k):
    scaler = StandardScaler().fit(population[utils.CLUSTER_FEATURES])
    X = scaler.transform(population[utils.CLUSTER_FEATURES])
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE).fit(X)
    raw_labels = kmeans.predict(X)
    # Нумерація за зростанням медіанної вартості портфеля
    medians = pd.Series(population["TOTAL_VEHICLE_VALUE_USD"].values).groupby(raw_labels).median()
    order = {int(raw): rank + 1 for rank, raw in enumerate(medians.sort_values().index)}
    return scaler, kmeans, order


def predict_clusters(population, scaler, kmeans, order):
    X = scaler.transform(population[utils.CLUSTER_FEATURES])
    return pd.Series(kmeans.predict(X), index=population.index).map(order)


if USE_SAVED_MODEL:
    EV_RULE = saved_model["ev_rule"]
    km_population = build_population(ev_rule=EV_RULE)
    scaler, kmeans, order = saved_model["scaler"], saved_model["kmeans"], saved_model["order"]
    km_population["CLUSTER_ID"] = predict_clusters(km_population, scaler, kmeans, order)
    print(f"Модель завантажено: K={K_CHOSEN}, EV_RULE={EV_RULE}")
else:
    # Крок 1: навчання на популяції з електромобілістами
    km_population = build_population(ev_rule=False)
    scaler, kmeans, order = fit_kmeans(km_population, K_CHOSEN)
    km_population["CLUSTER_ID"] = predict_clusters(km_population, scaler, kmeans, order)

    # Крок 2: чи є окремий EV-кластер
    ev_clients = km_population["EV_SHARE"].eq(1)
    ev_mean = km_population.groupby("CLUSTER_ID")["EV_SHARE"].mean()
    ev_cover = km_population.loc[ev_clients, "CLUSTER_ID"].value_counts(normalize=True)
    best = ev_mean.idxmax()
    has_ev_cluster = ev_mean.max() >= 0.8 and ev_cover.get(best, 0) >= 0.7
    print("Середній EV_SHARE по кластерах:")
    display(ev_mean.round(3).to_frame("MEAN_EV_SHARE").T)

    if has_ev_cluster:
        EV_RULE = False
        print(f"Окремий EV-кластер є (CLUSTER_{best}) → EV_ADOPTERS визначатиме кластеризація.")
    else:
        EV_RULE = True
        print("Окремого EV-кластера немає → EV_ADOPTERS задаємо правилом (EV_SHARE = 1), перенавчаємо KMeans без них.")
        km_population = build_population(ev_rule=True)
        scaler, kmeans, order = fit_kmeans(km_population, K_CHOSEN)
        km_population["CLUSTER_ID"] = predict_clusters(km_population, scaler, kmeans, order)

    joblib.dump(
        {"k": K_CHOSEN, "ev_rule": EV_RULE, "scaler": scaler, "kmeans": kmeans, "order": order,
         "features": utils.CLUSTER_FEATURES},
        OUTPUT_DIR / "kmeans_model.joblib",
    )
    print("Модель збережено: output/kmeans_model.joblib")

print(f"Популяція KMeans: {len(km_population):,} | K={K_CHOSEN} | EV_RULE={EV_RULE}")

In [ ]:
cluster_profiles = utils.profile_table(km_population.assign(CLUSTER=lambda d: "CLUSTER_" + d["CLUSTER_ID"].astype(str)), "CLUSTER")
cluster_profiles = cluster_profiles.loc[sorted(cluster_profiles.index, key=lambda s: int(s.split("_")[1]))]
cluster_profiles.to_csv(OUTPUT_DIR / "cluster_profiles.csv")
print("ПРОФІЛІ КЛАСТЕРІВ (надішли цю таблицю):")
display(cluster_profiles)

# Heatmap відхилень від середнього популяції
heat_cols = ["MED_TOTAL_VALUE", "MED_AGE", "PCT_2PLUS_CARS", "MEAN_PREMIUM_SHARE", "MEAN_SUV_SHARE",
             "MEAN_EV_SHARE", "MEAN_HYBRID_SHARE", "MEAN_GAS_SHARE"]
heat = cluster_profiles[heat_cols].astype(float)
heat = (heat - heat.mean()) / heat.std().replace(0, 1)
fig, ax = plt.subplots(figsize=(11, 0.5 * len(heat) + 1.5))
im = ax.imshow(heat.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(heat_cols)))
ax.set_xticklabels(heat_cols, rotation=30, ha="right")
ax.set_yticks(range(len(heat)))
ax.set_yticklabels(heat.index)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        ax.text(j, i, f"{cluster_profiles[heat_cols].iloc[i, j]:,.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(im, ax=ax, label="відхилення від середнього (z)")
ax.set_title("Профілі кластерів")
plt.tight_layout()
plt.show()

---
### ⏸ Кінець запуску 1
Надішли таблицю метрик з кроку 7 і профілі кластерів з кроку 8, отримай назви та заповни `CLUSTER_NAMES` нижче.

---
## 9. Назви кластерів

Ключ — номер кластера (1…K), значення — назва сегмента. Кілька кластерів можуть мати однакову назву —
тоді вони об'єднуються в один сегмент.

In [ ]:
CLUSTER_NAMES = {k: f"CLUSTER_{k}" for k in range(1, K_CHOSEN + 1)}   # тимчасові назви

# Після отримання назв заміни словник, наприклад:
# CLUSTER_NAMES = {
#     1: "BUDGET_LEGACY",
#     2: "CORE_MASS_MARKET_BUDGET",
#     ...
# }

missing = set(range(1, K_CHOSEN + 1)) - set(CLUSTER_NAMES)
assert not missing, f"Немає назв для кластерів: {sorted(missing)}"
clashes = set(CLUSTER_NAMES.values()) & (set(utils.RULE_SEGMENTS) - ({"EV_ADOPTERS"} if not EV_RULE else set()))
if clashes:
    print(f"УВАГА: назви збігаються з сегментами-правилами: {clashes}")

km_population["CLUSTER_SEGMENT"] = km_population["CLUSTER_ID"].map(CLUSTER_NAMES)
display(km_population["CLUSTER_SEGMENT"].value_counts().to_frame("CLIENTS"))

## 10. Дерево рішень: правила замість KMeans

In [ ]:
X_tree = km_population[utils.TREE_FEATURES]
y_tree = km_population["CLUSTER_SEGMENT"]
X_train, X_test, y_train, y_test = train_test_split(
    X_tree, y_tree, test_size=0.2, random_state=RANDOM_STATE, stratify=y_tree
)
min_leaf = max(50, int(0.002 * len(X_train)))

tree_results = {}
for depth in (4, 5):
    model = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=min_leaf, random_state=RANDOM_STATE)
    model.fit(X_train, y_train)
    tree_results[depth] = {
        "model": model,
        "FIDELITY_TEST": (model.predict(X_test) == y_test).mean(),
        "LEAVES": model.get_n_leaves(),
    }

display(pd.DataFrame({d: {k: v for k, v in r.items() if k != "model"} for d, r in tree_results.items()}).T
        .rename_axis("max_depth"))

if TREE_DEPTH is None:
    gain = tree_results[5]["FIDELITY_TEST"] - tree_results[4]["FIDELITY_TEST"]
    TREE_DEPTH_CHOSEN = 5 if gain > 0.02 else 4
else:
    TREE_DEPTH_CHOSEN = int(TREE_DEPTH)
tree_model = tree_results[TREE_DEPTH_CHOSEN]["model"]
print(f"Обрано depth = {TREE_DEPTH_CHOSEN}")

In [ ]:
tree_rules = utils.rules_from_tree(tree_model, utils.TREE_FEATURES)
rule_pred = utils.apply_rules(km_population, tree_rules)

unassigned = int(rule_pred.isna().sum())
FIDELITY = float((rule_pred == km_population["CLUSTER_SEGMENT"]).mean())
print(f"Правил після злиття й округлення: {len(tree_rules['rules'])}")
print(f"Без правила: {unassigned}")
print(f"Fidelity округлених правил до KMeans (уся популяція): {FIDELITY:.2%}")

per_segment = pd.DataFrame({
    "CLIENTS_KMEANS": km_population["CLUSTER_SEGMENT"].value_counts(),
    "CLIENTS_RULES": rule_pred.value_counts(),
    "RECALL": km_population.assign(OK=rule_pred.eq(km_population["CLUSTER_SEGMENT"]))
              .groupby("CLUSTER_SEGMENT")["OK"].mean().round(3),
}).fillna(0)
display(per_segment)

print("Матриця: KMeans (рядки) × правила (стовпці)")
display(pd.crosstab(km_population["CLUSTER_SEGMENT"], rule_pred.fillna("NONE")))

print()
print(utils.hierarchy_text(tree_rules, EV_RULE, BUSINESS_THRESHOLD))

## 11. Фінальні сегменти (за правилами, без KMeans)

In [ ]:
segments = utils.assign_final_segments(customers, tree_rules, EV_RULE, BUSINESS_THRESHOLD)

# Контроль: у популяції дерева сегмент = результат правил
check = segments.loc[km_population.index, "FINAL_SEGMENT"].eq(rule_pred)
assert check.all(), "Сегменти популяції дерева не збігаються з правилами"

sizes = segments.groupby(["FINAL_SEGMENT", "SEGMENT_SOURCE"]).size().rename("CLIENTS").reset_index()
sizes["PCT"] = (sizes["CLIENTS"] / len(segments) * 100).round(2)
sizes = sizes.sort_values("CLIENTS", ascending=False).reset_index(drop=True)
print(f"Клієнтів: {len(segments):,} | сума по сегментах: {sizes['CLIENTS'].sum():,} | сегментів: {len(sizes)}")
display(sizes)

fig, ax = plt.subplots(figsize=(9, 0.4 * len(sizes) + 1))
plot = sizes.sort_values("CLIENTS")
colors = plot["SEGMENT_SOURCE"].map({"RULE": "#8da0cb", "TREE": "#fc8d62"})
ax.barh(plot["FINAL_SEGMENT"], plot["CLIENTS"], color=colors)
for y, (value, pct) in enumerate(zip(plot["CLIENTS"], plot["PCT"])):
    ax.text(value, y, f" {value:,} ({pct}%)", va="center", fontsize=8)
ax.set_title("Розмір сегментів (синій — правило, помаранчевий — дерево)")
plt.tight_layout()
plt.show()

In [ ]:
segment_profiles = utils.profile_table(segments, "FINAL_SEGMENT")
segment_profiles.to_csv(OUTPUT_DIR / "segment_profiles.csv")
print("Профілі фінальних сегментів:")
display(segment_profiles)

## 12. Збереження результатів

In [ ]:
config = {
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "k": K_CHOSEN,
    "ev_rule": bool(EV_RULE),
    "business_threshold": BUSINESS_THRESHOLD,
    "age_median": AGE_MEDIAN,
    "reference_year": utils.REFERENCE_YEAR,
    "tree_depth": TREE_DEPTH_CHOSEN,
    "fidelity": round(FIDELITY, 4),
    "cluster_names": {str(k): v for k, v in CLUSTER_NAMES.items()},
}
utils.save_json(config, OUTPUT_DIR / "segmentation_config.json")
utils.save_json(tree_rules, OUTPUT_DIR / "tree_rules.json")
(OUTPUT_DIR / "segmentation_rules.txt").write_text(utils.hierarchy_text(tree_rules, EV_RULE, BUSINESS_THRESHOLD), encoding="utf-8")
(OUTPUT_DIR / "segmentation_rules.sql").write_text(utils.hierarchy_sql(tree_rules, EV_RULE, BUSINESS_THRESHOLD), encoding="utf-8")

result = segments[utils.output_columns(segments)].copy()
result["KMEANS_CLUSTER"] = km_population["CLUSTER_ID"].reindex(result.index)
result.to_csv(OUTPUT_DIR / "customer_segments.csv.gz", index=False, compression="gzip", encoding="utf-8-sig")

print(json.dumps(config, ensure_ascii=False, indent=2))
print("\nФайли в output/:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {path.name:35s} {path.stat().st_size / 1024:,.0f} KB")